# BERT + CRF Unified — Training Notebook (Kaggle)

**Unified Sequence Labeling**: 1 model duy nhất trích xuất aspect + sentiment đồng thời.

| File | Vai trò |
|------|--------- |
| `src/models/bert_crf/model.py` | Model: BERT + CRF (`num_labels=7`) |
| `src/training/train_unified.py` | Training script (hỗ trợ cả BERT + BiLSTM) |
| `src/data/unified_dataset.py` | Dataset + DataLoader |
| `configs/bert_unified_restaurants.yaml` | Config Kaggle |

### 7-label BIO scheme
| ID | Label | Ý nghĩa |
|----|-------|---------|
| 0 | O | Outside |
| 1 | B-POS | Begin Positive |
| 2 | I-POS | Inside Positive |
| 3 | B-NEG | Begin Negative |
| 4 | I-NEG | Inside Negative |
| 5 | B-NEU | Begin Neutral |
| 6 | I-NEU | Inside Neutral |

### Kỹ thuật
| Nhóm | Chi tiết |
|------|----------|
| Anti-overfitting | Dropout 0.3, Weight Decay 0.01, Label Smoothing 0.1 |
| Learning rate | Warmup + Cosine, LLRD (decay=0.9) |
| Decoding | CRF (transition matrix học joint label constraints) |

## 1. Cài đặt thư viện

In [1]:
!pip install -q pytorch-crf seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## 2. Clone GitHub Repo

In [2]:
# ==============================================================
# THÊM TOKEN VÀ LINK REPO TẠI ĐÂY
# ==============================================================
from kaggle_secrets import UserSecretsClient
GITHUB_TOKEN = UserSecretsClient().get_secret("GH_TOKEN")
GITHUB_REPO  = "sin0235/NLP"
BRANCH       = "main"
# ==============================================================

import subprocess, sys
from pathlib import Path

CLONE_DIR   = Path("/kaggle/working/repo")
WORKING_DIR = Path("/kaggle/working")

if CLONE_DIR.exists():
    result = subprocess.run(
        ["git", "-C", str(CLONE_DIR), "pull"],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
else:
    repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    result = subprocess.run(
        ["git", "clone", "--depth=1", "-b", BRANCH, repo_url, str(CLONE_DIR)],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )

print(result.stdout)
if result.returncode != 0:
    raise RuntimeError("Git clone/pull thất bại. Kiểm tra lại TOKEN và REPO.")

PROJECT_ROOT = CLONE_DIR
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

Cloning into '/kaggle/working/repo'...

Project root: /kaggle/working/repo


## 3. Kiểm tra Unified BIO Data

File unified BIO đã có sẵn trong Kaggle dataset — **không cần tạo lại**.

In [3]:
DATASET_INPUT = Path("/kaggle/input/datasets/luphuc/dataset-nlp-semeval-2014/DeBERTa")

for split in ["train", "val", "test"]:
    p = DATASET_INPUT / f"restaurants_{split}_unified_wordpiece.csv"
    status = f"OK  ({p.stat().st_size:,} bytes)" if p.exists() else "NOT FOUND"
    print(f"  {p.name}: {status}")

  restaurants_train_unified_wordpiece.csv: OK  (1,538,407 bytes)
  restaurants_val_unified_wordpiece.csv: OK  (182,394 bytes)
  restaurants_test_unified_wordpiece.csv: OK  (328,906 bytes)


In [4]:
import pandas as pd
print("\n[Số bản ghi train/val/test]")
for split in ["train", "val", "test"]:
    p = DATASET_INPUT / f"restaurants_{split}_unified_wordpiece.csv"
    if not p.exists():
        print(f"  {split:5s}: NOT FOUND — {p}")
        continue
    df = pd.read_csv(p)
    print(f"  {split:5s}: {len(df):,} records  |  file={p.name}")


[Số bản ghi train/val/test]
  train: 3,374 records  |  file=restaurants_train_unified_wordpiece.csv
  val  : 457 records  |  file=restaurants_val_unified_wordpiece.csv
  test : 800 records  |  file=restaurants_test_unified_wordpiece.csv


## 4. Cấu hình & Patch Config

In [5]:
import yaml

# ==============================================================
# CHỈNH RUN_ID KHI THỬ HYPERPARAMETERS MỚI
# ==============================================================
RUN_ID = "run01"
# ==============================================================

config_src  = "/kaggle/working/repo/configs/deberta_unified_restaurants.yaml"
config_dest = WORKING_DIR  / "deberta_restaurants_active.yaml"

with open(config_src) as f:
    cfg = yaml.safe_load(f)
    

# Data đọc từ Kaggle dataset (read-only)
cfg["data"]["train_path"] = "/kaggle/input/datasets/luphuc/dataset-nlp-semeval-2014/DeBERTa/restaurants_train_unified_wordpiece.csv"
cfg["data"]["val_path"]   = "/kaggle/input/datasets/luphuc/dataset-nlp-semeval-2014/DeBERTa/restaurants_val_unified_wordpiece.csv"
cfg["data"]["test_path"]  =  "/kaggle/input/datasets/luphuc/dataset-nlp-semeval-2014/DeBERTa/restaurants_test_unified_wordpiece.csv"

# Output → /kaggle/working/ (ghi được)
cfg["output"]["run_id"]          = RUN_ID
cfg["output"]["checkpoint_dir"]  = "/kaggle/working/outputs/checkpoints/bert_unified/"
cfg["output"]["log_dir"]         = "/kaggle/working/outputs/logs/bert_unified/"
cfg["output"]["result_dir"]      = "/kaggle/working/outputs/results/bert_unified/"
cfg["output"]["tensorboard_dir"] = "/kaggle/working/outputs/logs/bert_unified/tensorboard/"

for key in ["checkpoint_dir", "log_dir", "result_dir", "tensorboard_dir"]:
    Path(cfg["output"][key]).mkdir(parents=True, exist_ok=True)

with open(config_dest, "w") as f:
    yaml.dump(cfg, f, allow_unicode=True)

print(f"Config: {config_dest}  |  Run: {RUN_ID}")
for k in ["train_path", "val_path", "test_path"]:
    p = cfg["data"][k]
    print(f"  {k}: {'OK' if Path(p).exists() else 'NOT FOUND'} — {p}")

Config: /kaggle/working/deberta_restaurants_active.yaml  |  Run: run01
  train_path: OK — /kaggle/input/datasets/luphuc/dataset-nlp-semeval-2014/DeBERTa/restaurants_train_unified_wordpiece.csv
  val_path: OK — /kaggle/input/datasets/luphuc/dataset-nlp-semeval-2014/DeBERTa/restaurants_val_unified_wordpiece.csv
  test_path: OK — /kaggle/input/datasets/luphuc/dataset-nlp-semeval-2014/DeBERTa/restaurants_test_unified_wordpiece.csv


## 5. Hyperparameters (tuỳ chỉnh nếu cần)

Mọi giá trị đã có sẵn trong `configs/bert_unified_restaurants.yaml`.
Bỏ comment dòng nào muốn override.

In [6]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.training.train_unified import compute_class_weights_from_csv

weights = compute_class_weights_from_csv(
    cfg["data"]["train_path"],
    max_weight=10.0,
)
cfg["model_params"]["class_weights"] = weights

# ==============================================================
# BỎ COMMENT DÒNG NÀO MUỐN OVERRIDE (mặc định dùng YAML)
# ==============================================================

# cfg["model_params"]["pretrained_model"]    = "bert-base-uncased"
# cfg["training"]["num_epochs"]               = 80
# cfg["training"]["learning_rate"]            = 2e-5
# cfg["training"]["early_stopping_patience"]  = 8
# cfg["data"]["batch_size"]                   = 16

# ── Lưu config (luôn chạy) ───────────────────────────────────
with open(config_dest, "w") as f:
    yaml.dump(cfg, f, allow_unicode=True)

print("=== Active Hyperparameters ===")
for section in ["training", "data", "model_params"]:
    for k, v in cfg[section].items():
        print(f"  [{section}] {k}: {v}")


[Auto Class Weights] Tính từ training CSV:
  Label         Count     Weight
  --------------------------------
  O            57,934     0.1603
  B-POS         1,822     5.0963
  I-POS         1,120     8.2906
  B-NEG         1,639     5.6653
  I-NEG           432    10.0000
  B-NEU         1,585     5.8583
  I-NEU           466    10.0000
  class_weights: [0.1603, 5.0963, 8.2906, 5.6653, 10.0, 5.8583, 10.0]
=== Active Hyperparameters ===
  [training] num_epochs: 100
  [training] learning_rate: 2e-05
  [training] optimizer: adamw
  [training] weight_decay: 0.05
  [training] warmup_ratio: 0.2
  [training] gradient_clip: 1.0
  [training] gradient_accumulation_steps: 4
  [training] early_stopping_patience: 10
  [training] scheduler_type: cosine
  [training] use_llrd: True
  [training] llrd_decay: 0.95
  [training] use_amp: False
  [training] amp_dtype: bf16
  [training] lr_scheduler_factor: 0.5
  [training] lr_scheduler_patience: 3
  [training] plateau_mode: max
  [data] train_path: /kag

## 6. Chạy Training Script

In [7]:
import subprocess, os

train_script = str(PROJECT_ROOT / "src/training/train_unified.py")
cmd = ["python", "-u", train_script, "--config", str(config_dest)]
print(f"Chạy: {' '.join(cmd)}")
print("=" * 60)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=str(PROJECT_ROOT),
    env=env,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()
if return_code != 0:
    print(f"\nERROR: exit code {return_code}")
else:
    print("\n✓ Training hoàn thành!")

Chạy: python -u /kaggle/working/repo/src/training/train_unified.py --config /kaggle/working/deberta_restaurants_active.yaml
Unified Training — deberta_crf_unified
Device: cuda
Labels: ['O', 'B-POS', 'I-POS', 'B-NEG', 'I-NEG', 'B-NEU', 'I-NEU']

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 3065.65it/s, Materializing param=encoder.rel_embeddings.weight]
DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNE

## 7. Kết quả & Checkpoints

In [8]:
ckpt_dir = Path(cfg["output"]["checkpoint_dir"])
checkpoints = sorted(ckpt_dir.glob("*.pt"))

print(f"Checkpoints ({len(checkpoints)}):\n")
for p in checkpoints:
    print(f"  {p.name}  ({p.stat().st_size / 1024 / 1024:.1f} MB)")

if checkpoints:
    best = checkpoints[-1]
    print(f"\nBest checkpoint: {best.name}")

Checkpoints (14):

  deberta_crf_unified_restaurants_run01_epoch01_val_f1_0.0256.pt  (710.4 MB)
  deberta_crf_unified_restaurants_run01_epoch04_val_f1_0.3005.pt  (710.4 MB)
  deberta_crf_unified_restaurants_run01_epoch05_val_f1_0.4617.pt  (710.4 MB)
  deberta_crf_unified_restaurants_run01_epoch06_val_f1_0.5415.pt  (710.4 MB)
  deberta_crf_unified_restaurants_run01_epoch07_val_f1_0.5914.pt  (710.4 MB)
  deberta_crf_unified_restaurants_run01_epoch08_val_f1_0.6043.pt  (710.4 MB)
  deberta_crf_unified_restaurants_run01_epoch10_val_f1_0.6196.pt  (710.4 MB)
  deberta_crf_unified_restaurants_run01_epoch11_val_f1_0.6610.pt  (710.4 MB)
  deberta_crf_unified_restaurants_run01_epoch13_val_f1_0.6736.pt  (710.4 MB)
  deberta_crf_unified_restaurants_run01_epoch17_val_f1_0.6877.pt  (710.4 MB)
  deberta_crf_unified_restaurants_run01_epoch19_val_f1_0.6984.pt  (710.4 MB)
  deberta_crf_unified_restaurants_run01_epoch21_val_f1_0.7082.pt  (710.4 MB)
  deberta_crf_unified_restaurants_run01_epoch23_val_f1_0.